In [1]:
from google.colab import drive
import os

drive.mount("/content/drive")

RUTA = "/content/drive/MyDrive/radar-micro-Doppler/processed"
print(os.listdir(RUTA))

Mounted at /content/drive
['senales', 'espectrogramas_5s.npz', 'particion.csv', 'metadatos.csv', 'cnn_5s.pt']


In [2]:
import torch
print(torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

2.11.0+cu130
GPU disponible: True


In [3]:
import numpy as np
import pandas as pd

datos = np.load(os.path.join(RUTA, "espectrogramas_5s.npz"))

X = datos["X"]
ids = datos["ids"]

meta = pd.read_csv(os.path.join(RUTA, "metadatos.csv"))
part = pd.read_csv(os.path.join(RUTA, "particion.csv"))

df = meta.merge(part[["id", "particion"]], on="id")

assert (df["id"].values == ids).all(), "El orden de X y del CSV no coincide"

y = df["actividad"].values - 1

mask_train = (df["particion"] == "train").values
mask_valid = (df["particion"] == "valid").values
mask_test  = (df["particion"] == "test").values

X_train, y_train = X[mask_train], y[mask_train]
X_valid, y_valid = X[mask_valid], y[mask_valid]
X_test,  y_test  = X[mask_test],  y[mask_test]

print("train:", X_train.shape, np.bincount(y_train))
print("valid:", X_valid.shape, np.bincount(y_valid))
print("test: ", X_test.shape,  np.bincount(y_test))

train: (1224, 128, 128) [218 218 216 215 215 142]
valid: (253, 128, 128) [46 46 47 47 47 20]
test:  (277, 128, 128) [48 48 48 49 48 36]


In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def a_dataset(X, y):
  X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
  y_t = torch.tensor(y, dtype=torch.long)

  return TensorDataset(X_t, y_t)

ds_train = a_dataset(X_train, y_train)
ds_valid = a_dataset(X_valid, y_valid)
ds_test = a_dataset(X_test, y_test)

BATCH = 32
dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True)
dl_valid = DataLoader(ds_valid, batch_size=BATCH, shuffle=False)
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False)

imgs, etiquetas = next(iter(dl_train))

In [8]:
from torchvision import models

modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(modelo.fc)       # Linear(in_features=512, out_features=1000, bias=True)

for parametro in modelo.parameters():
  parametro.requires_grad = False

Linear(in_features=512, out_features=1000, bias=True)


In [10]:
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo.fc = nn.Linear(modelo.fc.in_features, 6)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.fc.parameters(), lr=1e-3)

In [ ]:
class ResNetEspectro(nn.Module):
    def __init__(self, modelo):
        super().__init__()
        self.modelo = modelo
        # Media y desviación de ImageNet, con forma (1, 3, 1, 1) para operar por canal
        self.register_buffer("media", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",   torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)            # (N, 1, 128, 128) → (N, 3, 128, 128)
        x = (x - self.media) / self.std      # misma normalización que en ImageNet
        return self.modelo(x)

modelo = ResNetEspectro(modelo).to(device)

In [12]:
import copy

historial = {"loss_train": [], "loss_valid": [], "acc_train": [], "acc_valid": []}
MODELOS = []
validation_error = []
EPOCAS = 30

for epoca in range(EPOCAS):

    # ---------- FASE DE ENTRENAMIENTO ----------
    modelo.train()
    total_loss, aciertos, n = 0.0, 0, 0

    for imgs, etiquetas in dl_train:
        imgs, etiquetas = imgs.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = modelo(imgs)
        loss = criterio(salidas, etiquetas)
        loss.backward()
        optimizador.step()

        total_loss += loss.item() * imgs.size(0)
        aciertos += (salidas.argmax(1) == etiquetas).sum().item()
        n += imgs.size(0)

    historial["loss_train"].append(total_loss / n)
    historial["acc_train"].append(aciertos / n)

    # ---------- FASE DE VALIDACIÓN ----------
    modelo.eval()
    total_loss, aciertos, n = 0.0, 0, 0

    with torch.no_grad():
        for imgs, etiquetas in dl_valid:
            imgs, etiquetas = imgs.to(device), etiquetas.to(device)

            salidas = modelo(imgs)
            loss = criterio(salidas, etiquetas)

            total_loss += loss.item() * imgs.size(0)
            aciertos += (salidas.argmax(1) == etiquetas).sum().item()
            n += imgs.size(0)

    validation_error.append(total_loss / n)
    MODELOS.append(copy.deepcopy(modelo.state_dict()))

    historial["loss_valid"].append(total_loss / n)
    historial["acc_valid"].append(aciertos / n)

    print(f"Época {epoca+1:2d} | "
          f"train loss {historial['loss_train'][-1]:.3f} acc {historial['acc_train'][-1]:.3f} | "
          f"valid loss {historial['loss_valid'][-1]:.3f} acc {historial['acc_valid'][-1]:.3f}")

mejor = np.argmin(validation_error)
modelo.load_state_dict(MODELOS[mejor])
print(f"Mejor época: {mejor + 1} (valid loss {validation_error[mejor]:.3f})")

Época  1 | train loss 1.437 acc 0.443 | valid loss 1.585 acc 0.336
Época  2 | train loss 0.966 acc 0.686 | valid loss 0.990 acc 0.640
Época  3 | train loss 0.804 acc 0.743 | valid loss 0.909 acc 0.684
Época  4 | train loss 0.707 acc 0.775 | valid loss 0.875 acc 0.708
Época  5 | train loss 0.656 acc 0.785 | valid loss 0.847 acc 0.688
Época  6 | train loss 0.608 acc 0.809 | valid loss 0.807 acc 0.723
Época  7 | train loss 0.568 acc 0.823 | valid loss 0.796 acc 0.731
Época  8 | train loss 0.557 acc 0.818 | valid loss 0.777 acc 0.747
Época  9 | train loss 0.533 acc 0.840 | valid loss 0.768 acc 0.739
Época 10 | train loss 0.514 acc 0.830 | valid loss 0.738 acc 0.755
Época 11 | train loss 0.478 acc 0.842 | valid loss 0.737 acc 0.747
Época 12 | train loss 0.465 acc 0.856 | valid loss 0.714 acc 0.755
Época 13 | train loss 0.462 acc 0.851 | valid loss 0.730 acc 0.759
Época 14 | train loss 0.421 acc 0.873 | valid loss 0.709 acc 0.751
Época 15 | train loss 0.423 acc 0.867 | valid loss 0.695 acc 0

In [14]:
y_real, y_pred = [], []

modelo.eval()
with torch.no_grad():
    for imgs, etiquetas in dl_test:
        imgs = imgs.to(device)
        salidas = modelo(imgs)

        y_pred.extend(salidas.argmax(1).cpu().numpy())
        y_real.extend(etiquetas.numpy())

y_real = np.array(y_real)
y_pred = np.array(y_pred)
print(len(y_real), len(y_pred))

277 277


In [15]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

nombres = ["andar", "sentarse", "levantarse", "agacharse", "beber", "caerse"]

print("Precisión test:", accuracy_score(y_real, y_pred))
print(confusion_matrix(y_real, y_pred))
print(classification_report(y_real, y_pred, target_names=nombres, digits=3))

Precisión test: 0.8303249097472925
[[46  0  1  0  0  1]
 [ 2 37  4  4  1  0]
 [ 0  0 41  3  3  1]
 [ 3  1  3 39  3  0]
 [ 1  3  2 10 32  0]
 [ 0  0  0  0  1 35]]
              precision    recall  f1-score   support

       andar      0.885     0.958     0.920        48
    sentarse      0.902     0.771     0.831        48
  levantarse      0.804     0.854     0.828        48
   agacharse      0.696     0.796     0.743        49
       beber      0.800     0.667     0.727        48
      caerse      0.946     0.972     0.959        36

    accuracy                          0.830       277
   macro avg      0.839     0.836     0.835       277
weighted avg      0.834     0.830     0.829       277



In [18]:
from torchvision import models

modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
modelo.fc = nn.Linear(modelo.fc.in_features, 6)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-4)

In [19]:
modelo = ResNetEspectro(modelo).to(device)

In [20]:
print(sum(p.numel() for p in modelo.parameters() if p.requires_grad))   # ~11 179 590

11179590


In [21]:
import copy

historial = {"loss_train": [], "loss_valid": [], "acc_train": [], "acc_valid": []}
MODELOS = []
validation_error = []
EPOCAS = 30

for epoca in range(EPOCAS):

    # ---------- FASE DE ENTRENAMIENTO ----------
    modelo.train()
    total_loss, aciertos, n = 0.0, 0, 0

    for imgs, etiquetas in dl_train:
        imgs, etiquetas = imgs.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = modelo(imgs)
        loss = criterio(salidas, etiquetas)
        loss.backward()
        optimizador.step()

        total_loss += loss.item() * imgs.size(0)
        aciertos += (salidas.argmax(1) == etiquetas).sum().item()
        n += imgs.size(0)

    historial["loss_train"].append(total_loss / n)
    historial["acc_train"].append(aciertos / n)

    # ---------- FASE DE VALIDACIÓN ----------
    modelo.eval()
    total_loss, aciertos, n = 0.0, 0, 0

    with torch.no_grad():
        for imgs, etiquetas in dl_valid:
            imgs, etiquetas = imgs.to(device), etiquetas.to(device)

            salidas = modelo(imgs)
            loss = criterio(salidas, etiquetas)

            total_loss += loss.item() * imgs.size(0)
            aciertos += (salidas.argmax(1) == etiquetas).sum().item()
            n += imgs.size(0)

    validation_error.append(total_loss / n)
    MODELOS.append(copy.deepcopy(modelo.state_dict()))

    historial["loss_valid"].append(total_loss / n)
    historial["acc_valid"].append(aciertos / n)

    print(f"Época {epoca+1:2d} | "
          f"train loss {historial['loss_train'][-1]:.3f} acc {historial['acc_train'][-1]:.3f} | "
          f"valid loss {historial['loss_valid'][-1]:.3f} acc {historial['acc_valid'][-1]:.3f}")

mejor = np.argmin(validation_error)
modelo.load_state_dict(MODELOS[mejor])
print(f"Mejor época: {mejor + 1} (valid loss {validation_error[mejor]:.3f})")

Época  1 | train loss 0.629 acc 0.771 | valid loss 0.920 acc 0.708
Época  2 | train loss 0.206 acc 0.929 | valid loss 0.379 acc 0.874
Época  3 | train loss 0.119 acc 0.960 | valid loss 0.550 acc 0.810
Época  4 | train loss 0.100 acc 0.971 | valid loss 0.462 acc 0.862
Época  5 | train loss 0.077 acc 0.975 | valid loss 0.583 acc 0.858
Época  6 | train loss 0.079 acc 0.975 | valid loss 0.869 acc 0.854
Época  7 | train loss 0.044 acc 0.987 | valid loss 0.349 acc 0.905
Época  8 | train loss 0.035 acc 0.990 | valid loss 0.604 acc 0.846
Época  9 | train loss 0.028 acc 0.993 | valid loss 0.300 acc 0.897
Época 10 | train loss 0.013 acc 0.998 | valid loss 0.346 acc 0.885
Época 11 | train loss 0.022 acc 0.993 | valid loss 0.496 acc 0.866
Época 12 | train loss 0.021 acc 0.994 | valid loss 0.395 acc 0.877
Época 13 | train loss 0.026 acc 0.994 | valid loss 0.572 acc 0.881
Época 14 | train loss 0.051 acc 0.983 | valid loss 0.711 acc 0.877
Época 15 | train loss 0.045 acc 0.984 | valid loss 0.641 acc 0

In [22]:
y_real, y_pred = [], []

modelo.eval()
with torch.no_grad():
    for imgs, etiquetas in dl_test:
        imgs = imgs.to(device)
        salidas = modelo(imgs)

        y_pred.extend(salidas.argmax(1).cpu().numpy())
        y_real.extend(etiquetas.numpy())

y_real = np.array(y_real)
y_pred = np.array(y_pred)
print(len(y_real), len(y_pred))

277 277


In [23]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

nombres = ["andar", "sentarse", "levantarse", "agacharse", "beber", "caerse"]

print("Precisión test:", accuracy_score(y_real, y_pred))
print(confusion_matrix(y_real, y_pred))
print(classification_report(y_real, y_pred, target_names=nombres, digits=3))

Precisión test: 0.9530685920577617
[[48  0  0  0  0  0]
 [ 0 48  0  0  0  0]
 [ 0  0 47  0  1  0]
 [ 0  1  2 45  1  0]
 [ 0  1  0  7 40  0]
 [ 0  0  0  0  0 36]]
              precision    recall  f1-score   support

       andar      1.000     1.000     1.000        48
    sentarse      0.960     1.000     0.980        48
  levantarse      0.959     0.979     0.969        48
   agacharse      0.865     0.918     0.891        49
       beber      0.952     0.833     0.889        48
      caerse      1.000     1.000     1.000        36

    accuracy                          0.953       277
   macro avg      0.956     0.955     0.955       277
weighted avg      0.954     0.953     0.953       277



In [25]:
torch.save(modelo.state_dict(), os.path.join(RUTA, "resnet18_finetune_5s.pt"))